In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.stores as
select *, 
  current_timestamp() as ingested_at,
  _metadata.file_path as _source_file
  from read_files("/Volumes/{catalog}/landing/raw/stores/stores.csv")
""")

In [0]:
%sql
drop table if exists fk_ce1366.bronze.customers

In [0]:
spark.sql(f"""
create table if not exists {catalog}.bronze.customers
""")

spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM "/Volumes/{catalog}/landing/raw/customers/"
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/ops/pipeline_state/products_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudfiles.inferColumnTypes", "true")
    .option("multiline", "true")
    .load(f"/Volumes/{catalog}/landing/raw/products/")
    .select(
            "*",
            col("_metadata.file_path").alias("_source_file"),
            current_timestamp().alias("ingested_at")
        )
    .writeStream
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/products_checkpoint")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.products")
  )


In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/ops/pipeline_state/order_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudfiles.inferColumnTypes", "true")
    .option("multiline", "true")
    .load(f"/Volumes/{catalog}/landing/raw/orders/")
    .select(
            "*",
            col("_metadata.file_path").alias("_source_file"),
            current_timestamp().alias("ingested_at")
        )
    .writeStream
    .option("checkpointLocation", f"/Volumes/{catalog}/ops/pipeline_state/orders_checkpoint")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.orders")
  )


In [0]:
%sql
select * from fk_ce1366.bronze.orders


In [0]:
%sql
select * from fk_ce1366.bronze.products

In [0]:
%sql
select * from fk_ce1366.bronze.customers

In [0]:
%sql
select * from fk_ce1366.bronze.stores